# PM Skills with DeepSeek

Run professional skills (weekly reports, PRDs, lease and contract decoders) on DeepSeek's API.

## 1. Install

The `pm-skills` package bundles the skills as data; nothing else is needed (the model call below uses only the standard library).

In [ ]:
%pip install -q pm-skills

## 2. Connect

Set `DEEPSEEK_API_KEY` in your environment (from platform.deepseek.com). The key is read from the environment, never written into the notebook.

In [ ]:
import json, os, urllib.request

def chat(base_url, model, system, user, key=None, timeout=120):
    """One chat completion from any OpenAI-compatible endpoint, standard library only."""
    body = json.dumps({"model": model, "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}).encode()
    headers = {"content-type": "application/json"}
    if key:
        headers["authorization"] = "Bearer " + key
    req = urllib.request.Request(base_url.rstrip("/") + "/chat/completions", data=body, headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)["choices"][0]["message"]["content"]

BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-chat"
API_KEY = os.environ.get("DEEPSEEK_API_KEY")
assert API_KEY, "Set DEEPSEEK_API_KEY first"

## 3. Find the right skill

`search_skills` ranks skills by how well their name and description match your words.

In [ ]:
import pm_skills

for s in pm_skills.search_skills("weekly report", limit=5):
    print(s["name"], "-", s["title"])

## 4. Run a skill

`skill_prompt` returns the skill's full instructions; send them as the system prompt and your material as the user message.

In [ ]:
notes = """Shipped the new checkout to 20% of traffic, conversion up 1.8%.
Payment provider timeouts twice this week, ticket raised.
Next week: full rollout, review the test."""
print(chat(BASE_URL, MODEL, pm_skills.skill_prompt("cn-weekly-report"), notes, key=API_KEY))

## 5. Route automatically

Let the search pick the skill from a plain-language request, then run it.

In [ ]:
request = "my landlord kept my deposit for cleaning, what can I do"
best = pm_skills.search_skills("deposit", limit=1)[0]
print("Using:", best["name"])
print(chat(BASE_URL, MODEL, pm_skills.skill_prompt(best["name"]), request, key=API_KEY)[:1500])

Skills: https://github.com/mohitagw15856/pm-claude-skills · The output is a draft for you to check, not professional advice.